In [4]:
%pip install tiktoken litellm

Note: you may need to restart the kernel to use updated packages.


In [17]:
import litellm
import tiktoken
import json

CACHED_REQUESTS = 50
TOKENS_PER_MILLION = 1_000_000
CLAUDE_45_SONNET_MODEL = "claude-sonnet-4-5-20250929"


def _count_tokens(text: str, model: str) -> int:
    """Count tokens in raw text using the model's tokenizer."""
    return len(tiktoken.encoding_for_model(model).encode(text))


def cost_gpt_5(text: str) -> float:
    """Return the USD cost of one input and 50 cached GPT-5 reuses."""
    tokens = _count_tokens(text, "gpt-5")
    input_cost = 1.25 / TOKENS_PER_MILLION
    input_cache_cost = 0.125 / TOKENS_PER_MILLION
    cost = tokens * input_cost + CACHED_REQUESTS * tokens * input_cache_cost
    return cost


def cost_gpt_4o(text: str) -> float:
    """Return the USD cost of one input and 50 cached GPT-4o reuses."""
    tokens = _count_tokens(text, "gpt-4o")
    input_cost = 2.50 / TOKENS_PER_MILLION
    input_cache_cost = 1.25 / TOKENS_PER_MILLION
    cost = tokens * input_cost + CACHED_REQUESTS * tokens * input_cache_cost
    return cost


def _count_claude_45_sonnet_tokens(text: str) -> int:
    """Estimate Claude tokens locally without requiring API credentials."""
    return litellm.utils.token_counter(
        text=text, model=CLAUDE_45_SONNET_MODEL
    )


def cost_claude_45_sonnet(text: str) -> float:
    """Return the USD cost of one 5m cache write and 50 cache reads."""
    tokens = _count_claude_45_sonnet_tokens(text)
    cache_write_cost = 3.75 / TOKENS_PER_MILLION
    cache_read_cost = 0.30 / TOKENS_PER_MILLION
    cost = tokens * cache_write_cost + CACHED_REQUESTS * tokens * cache_read_cost
    return cost

def report_ds(dir: str, model: str):
    with open(dir) as f:
        ds = [json.loads(i) for i in f]
    if model=="gpt-4o":
        func=cost_gpt_4o
    elif model=="gpt-5":
        func=cost_gpt_5
    elif model=="claude-45":
        func=cost_claude_45_sonnet
    repro = sum([func(str(i["F2P_content"])) for i in ds])/len(ds)
    regre = sum([func(str(i.get("test_cmd") or i.get("test_cmds"))+str(i["PASS_TO_PASS"])) for i in ds])/len(ds)
    loc = sum([func(str(i["location_content"])+str(i["location"])) for i in ds])/len(ds)
    con = sum([func(str(i["error_context"])) for i in ds])/len(ds)
    api = sum([func(str(i["api"])) for i in ds])/len(ds)
    print(
        round(repro,2),
        round(regre,2),
        round(loc,2),
        round(con,2),
        round(api,2),
        sep="\t"
    )

In [18]:
report_ds("/home/v-kenanli/Oracle-SWE/data/verified/7_test_loc_context_api_injected.jsonl", "gpt-4o")
report_ds("/home/v-kenanli/Oracle-SWE/data/verified/7_test_loc_context_api_injected.jsonl", "gpt-5")
report_ds("/home/v-kenanli/Oracle-SWE/data/live/6_test_loc_context_api_injected.jsonl", "gpt-5")
report_ds("/home/v-kenanli/Oracle-SWE/data/live/6_test_loc_context_api_injected.jsonl", "claude-45")
report_ds("/home/v-kenanli/Oracle-SWE/data/pro/pyt/6_test_loc_context_api_injected.jsonl", "gpt-5")
report_ds("/home/v-kenanli/Oracle-SWE/data/pro/pyt/6_test_loc_context_api_injected.jsonl", "claude-45")
report_ds("/home/v-kenanli/Oracle-SWE/data/pro/go/5_test_loc_context_api.jsonl", "gpt-5")

0.06	0.16	0.13	0.6	0.05
0.01	0.02	0.02	0.07	0.01
0.01	0.49	0.03	0.12	0.01
0.03	1.56	0.07	0.33	0.02
0.02	0.01	0.04	0.14	0.01
0.05	0.04	0.11	0.39	0.04
0.04	0.0	0.13	0.02	0.06
